# How Good Can This Model Possibly Get?

A churn model with a test ROC-AUC of 0.72 invites an obvious question: is that
a good score, or is there another 0.15 sitting on the table waiting for better
features?

Normally you cannot answer that. Here you can, because the data is synthetic
and the process that generated it is known. Every customer was assigned a
hidden `activity_level` that drives their order rate and browsing rate, and
that column was dropped before the CSVs were written.

So we can build an **oracle**: a model that sees the hidden driver directly.
Its score is the ceiling. Nothing built from observed behaviour can beat a
model that already knows the answer behaviour is a noisy measurement of.

The gap between the shipped model and the oracle is the real headroom. What is
left below the oracle is irreducible: whether a given customer happens to place
an order in a specific 30-day window is mostly a coin flip, however well we
know their propensity.

## 1. Setup

In [1]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.features import build_features
from src.generate_data import generate
from src.scoring import (
    CATEGORICAL_FEATURES,
    DERIVED_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_FEATURES,
    add_rate_features,
)

RANDOM_STATE = 42

## 2. Recover the hidden driver

`generate()` returns the customer frame with `activity_level` still attached.
It is seeded, so this reproduces exactly the same customers that are in
`data/customers.csv` -- the only difference is the column that was dropped on
the way to disk.

In [2]:
customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")
events = pd.read_csv("../data/website_events.csv")

hidden, _, _ = generate()

assert (hidden["customer_id"].to_numpy() == customers["customer_id"].to_numpy()).all()

features = build_features(customers, orders, events).merge(
    hidden[["customer_id", "activity_level"]],
    on="customer_id",
    how="left",
)

y = features["churn"]

print(f"eligible customers : {len(features):,}")
print(f"churn rate         : {y.mean():.4f}")
print(f"activity_level     : "
      f"min {features['activity_level'].min():.3f}, "
      f"max {features['activity_level'].max():.3f}")

eligible customers : 4,156
churn rate         : 0.6809
activity_level     : min 0.016, max 0.995


## 3. A single evaluation harness

Every variant below uses identical hyperparameters and an identical split, so
the only thing that changes is the feature set. Hyperparameters are fixed
rather than tuned per variant: tuning each one separately would confound
"better features" with "luckier search".

In [3]:
def split(X):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
    )
    return X_train, y_train, X_test, y_test


def evaluate(X, numeric, categorical=(), derive=False, label=""):
    X_train, y_train, X_test, y_test = split(X)

    steps = []
    if derive:
        steps.append(("rates", FunctionTransformer(add_rate_features)))

    transformers = [
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), list(numeric)),
    ]
    if categorical:
        transformers.append(
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), list(categorical))
        )

    steps += [
        ("preprocessor", ColumnTransformer(transformers)),
        ("model", GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            n_estimators=100,
        )),
    ]

    pipe = Pipeline(steps)
    pipe.fit(X_train, y_train)

    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"{label:<44} {auc:.4f}")
    return auc

## 4. The ladder

In [4]:
print(f"{'feature set':<44} {'test ROC-AUC'}")
print("-" * 60)

raw_only = evaluate(
    features[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
    NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    label="raw counts only",
)

shipped = evaluate(
    features[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
    MODEL_NUMERIC_FEATURES, CATEGORICAL_FEATURES, derive=True,
    label="shipped (raw + derived rates)",
)

oracle = evaluate(
    features[["activity_level"]], ["activity_level"],
    label="ORACLE: hidden activity_level alone",
)

oracle_all = evaluate(
    features[NUMERIC_FEATURES + ["activity_level"] + CATEGORICAL_FEATURES],
    NUMERIC_FEATURES + ["activity_level"], CATEGORICAL_FEATURES,
    label="ORACLE + observed features",
)

print()
print(f"headroom from raw counts to the ceiling : {oracle - raw_only:+.4f}")
print(f"closed by the derived rate features     : {shipped - raw_only:+.4f} "
      f"({(shipped - raw_only) / (oracle - raw_only):.0%} of the gap)")
print(f"remaining headroom                      : {oracle - shipped:+.4f}")

feature set                                  test ROC-AUC
------------------------------------------------------------


raw counts only                              0.7139


shipped (raw + derived rates)                0.7195
ORACLE: hidden activity_level alone          0.7575


ORACLE + observed features                   0.7611

headroom from raw counts to the ceiling : +0.0436
closed by the derived rate features     : +0.0056 (13% of the gap)
remaining headroom                      : +0.0380


## 5. Why the rate features help

The hidden driver is a *rate*. A raw count is that rate multiplied by how long
the customer has been around, so `total_orders` conflates "how keen are they"
with "how long have they had the chance". Dividing by tenure separates the two.

The correlations below show it directly: per-day rates track the hidden driver
more closely than the raw counts they are built from.

In [5]:
enriched = add_rate_features(features)

rows = []
for column in ["total_orders", "orders_per_day",
               "total_events", "events_per_day",
               "days_since_last_order", "recency_ratio",
               "tenure_days"]:
    rows.append({
        "feature": column,
        "spearman_vs_hidden": enriched[column].corr(
            enriched["activity_level"], method="spearman"
        ),
    })

pd.DataFrame(rows).set_index("feature").round(4)

,spearman_vs_hidden
feature,
total_orders,0.6223
orders_per_day,0.7288
total_events,0.6085
events_per_day,0.8060
days_since_last_order,-0.4681
recency_ratio,-0.4318
tenure_days,-0.0085


Note `tenure_days` sits near zero on its own. That is expected and correct: how
long someone has been a customer says nothing about how active they are, since
signup dates are drawn independently of `activity_level`. Tenure matters as a
*denominator*, not as a predictor.

An earlier version of the generator got this wrong. It assigned each order a
date drawn uniformly across the whole calendar year, independently of when the
customer signed up, which put 48% of orders before their customer existed. In
that dataset the rate features were actively harmful -- `orders_per_day` was a
worse signal than `total_orders`, because dividing by an unrelated tenure only
added noise.

## 6. What this means

The ceiling is not 1.0, and it is not even 0.85. Knowing every customer's true
propensity *perfectly* still only scores around 0.76, because the target asks
something inherently noisy: not "is this customer disengaging" but "will this
specific customer happen to place an order in these specific 30 days".

Two consequences worth stating plainly:

1. **Further feature engineering has little room to work.** The shipped model
   sits within a few points of the oracle. Effort spent on more behavioural
   features would be chasing a small remainder.

2. **The way to raise the ceiling is to change the target, not the model.** A
   longer target window, or a definition based on sustained disengagement
   rather than a single 30-day gap, would carry more signal per label. That is
   a problem-framing change, and it dominates anything available on the
   modelling side.

This is also why the project reports ROC-AUC rather than accuracy as its
headline: at a 68% base rate, ranking is the thing the model does well and the
thing the baseline cannot do at all.